# Credit Risk Analysis & Probability of Default (PD) Model

## 📋 **Project Overview**

This notebook demonstrates a comprehensive **Credit Risk Analysis** and builds a **Probability of Default (PD) Model** using machine learning techniques. 

### 🎯 **Objectives:**
- Analyze credit risk dataset and identify key risk factors
- Build predictive models to estimate probability of loan default
- Compare different machine learning algorithms (Logistic Regression, Random Forest, XGBoost)
- Generate risk predictions for portfolio management

### 🔍 **Business Context:**
In banking and finance, **Credit Risk** is the possibility that a borrower will fail to repay a loan. Accurate PD models help:
- Make informed lending decisions
- Set appropriate interest rates based on risk
- Calculate regulatory capital requirements
- Manage portfolio risk exposure

### 📊 **Dataset Information:**
- **Source**: Credit risk dataset with borrower and loan characteristics
- **Target Variable**: `loan_status` (0 = Good/Repaid, 1 = Default)
- **Features**: Demographics, employment, loan details, credit history

---

## 🔧 **Step 1: Import Essential Libraries**

We start by importing the necessary Python libraries for data manipulation, visualization, and machine learning.

### **Libraries Used:**
- **pandas**: Data manipulation and analysis
- **numpy**: Numerical computations
- **seaborn & matplotlib**: Data visualization
- **sklearn**: Machine learning algorithms and preprocessing

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [3]:
df=pd.read_csv('D:\Creditrisk_ENDtoEND\Rawdata\credit_risk_dataset.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
df.columns

Index(['person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length'],
      dtype='object')

## 📊 **Step 2: Data Loading and Initial Exploration** 

Let's load our credit risk dataset and perform initial exploration to understand the data structure.

### **What we're doing:**
1. Load the CSV file containing credit risk data
2. Display the first few rows to understand the data structure
3. Check column names and data types

In [5]:
credit_risk = pd.read_csv(r"D:\Creditrisk_ENDtoEND\Rawdata\credit_risk_dataset.csv")
credit_risk.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [6]:

credit_risk.describe()


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


### **📈 Statistical Summary**

The `describe()` function provides key statistical measures:
- **Count**: Number of non-null values
- **Mean**: Average value  
- **Std**: Standard deviation (measure of spread)
- **Min/Max**: Range of values
- **25%, 50%, 75%**: Quartiles (percentiles)

This helps identify potential outliers and understand data distributions.

In [7]:
credit_risk.pivot_table(index='person_age',columns='loan_status',values='person_income',aggfunc='count').reset_index().sort_values(by='person_age',ascending=False)

loan_status,person_age,0,1
57,144,3.0,NaN
56,123,2.0,NaN
55,94,1.0,NaN
54,84,1.0,NaN
53,80,1.0,NaN
52,78,1.0,NaN
51,76,1.0,NaN
50,73,3.0,NaN
49,70,5.0,2.0
48,69,5.0,NaN


### **🔍 Age vs Loan Status Analysis**

This pivot table shows the relationship between `person_age` and `loan_status`:
- **Rows**: Different age values
- **Columns**: Loan status (0=Good, 1=Default)  
- **Values**: Count of loans for each age-status combination

**Purpose**: Identify if age patterns exist in loan defaults. This analysis often reveals that very young or very old borrowers might have different risk profiles.

This code creates a new DataFrame (cr_age_rmvd) containing only records where person_age ≤ 70, resets its index to start from 0, and displays the first 5 rows.

## 🧹 **Step 3: Data Cleaning & Outlier Treatment**

### **Why Remove Age Outliers?**

From our analysis, we found some extreme age values that might be:
- **Data entry errors** (unrealistic ages)
- **Statistical outliers** that could skew our model

**Business Logic**: Most active borrowers are typically under 70 years old. Ages above this threshold might represent:
- Retired individuals with different risk profiles
- Potential data quality issues

In [8]:
cr_age_rmvd = credit_risk[credit_risk['person_age']<=70]
cr_age_rmvd.reset_index(drop=True, inplace=True)
cr_age_rmvd.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [9]:
credit_risk.shape

(32581, 12)

In [10]:
cr_age_rmvd.shape

(32568, 12)

### **📊 Impact of Age Filtering**

Let's compare the dataset sizes before and after age filtering to understand the impact:
- **Original dataset**: Shows total number of records
- **After age filtering**: Shows remaining records after removing age > 70

**Key Metrics to Monitor:**
- Number of records removed
- Percentage of data retained
- Impact on default rate distribution

removing person emp length

### **👔 Employment Length Outlier Treatment**

**Why Remove Employment Length Outliers?**

Similar to age, extremely high employment lengths (>47 years) might indicate:
- **Data quality issues**: Impossible employment durations
- **Career outliers**: People who started working very young
- **Data entry errors**: Misreported or incorrectly calculated values

**Business Rationale**: 
- Most careers span 20-40 years realistically
- Values >47 years likely represent data anomalies
- Removing these improves model reliability

In [11]:
person_emp_rmvd = cr_age_rmvd[cr_age_rmvd['person_emp_length']<=47]
person_emp_rmvd.reset_index(drop=True, inplace=True)
person_emp_rmvd.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


In [12]:
person_emp_rmvd.shape

(31671, 12)

In [13]:
person_emp_rmvd.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,31671.000000,3.167100e+04,31671.000000,31671.000000,28626.000000,31671.000000,31671.000000,31671.000000
mean,27.717754,6.649231e+04,4.780714,9660.637492,11.040070,0.215497,0.169621,5.804395
std,6.159859,5.277413e+04,4.028718,6334.716643,3.229507,0.411173,0.106275,4.048776
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.936600e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.600000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,8.000000e+04,7.000000,12500.000000,13.480000,0.000000,0.230000,8.000000
max,70.000000,2.039784e+06,38.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [14]:
person_emp_rmvd.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length                0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3045
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

## 🔧 **Step 4: Missing Value Treatment**

### **Identifying Missing Values**

Before building our model, we need to handle missing values as they can:
- **Reduce dataset size** if we remove records
- **Introduce bias** if missing values follow patterns
- **Break machine learning algorithms** that can't handle NaN values

**Common Strategies:**
- **Remove rows**: If few missing values
- **Imputation**: Fill with mean, median, or mode
- **Forward/Backward fill**: For time series data

In [15]:
cr_data = person_emp_rmvd.copy()

In [16]:
cr_data.fillna({'loan_int_rate':cr_data['loan_int_rate'].median()},inplace=True)

### **💡 Median Imputation for Interest Rate**

**Why use Median instead of Mean?**

For the `loan_int_rate` field, we choose **median imputation** because:

- **Robust to outliers**: Median is less affected by extreme values
- **Realistic values**: Interest rates might have outliers (very high or low rates)
- **Business logic**: Median represents a "typical" interest rate better than mean

**Alternative approaches:**
- Mean imputation (sensitive to outliers)
- Mode imputation (most frequent value)
- Predictive imputation (using other features)

In [17]:
cr_data.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

In [18]:
cr_data.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,31671.000000,3.167100e+04,31671.000000,31671.000000,31671.000000,31671.000000,31671.000000,31671.000000
mean,27.717754,6.649231e+04,4.780714,9660.637492,11.035256,0.215497,0.169621,5.804395
std,6.159859,5.277413e+04,4.028718,6334.716643,3.070364,0.411173,0.106275,4.048776
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.936600e+04,2.000000,5000.000000,8.490000,0.000000,0.090000,3.000000
50%,26.000000,5.600000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,8.000000e+04,7.000000,12500.000000,13.160000,0.000000,0.230000,8.000000
max,70.000000,2.039784e+06,38.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


In [19]:
cr_data.groupby('loan_status').count()['person_age']

loan_status
0    24846
1     6825
Name: person_age, dtype: int64

## 📈 **Step 5: Target Variable Analysis**

### **Understanding the Default Rate**

Let's analyze our target variable `loan_status`:
- **0**: Good loans (repaid successfully)  
- **1**: Default loans (failed to repay)

**Key Metrics:**
- **Total good loans**: Count of successful repayments
- **Total defaults**: Count of failed loans
- **Default rate**: Percentage of loans that defaulted

**Why this matters:**
- Helps understand class imbalance
- Informs business risk appetite
- Guides model evaluation metrics

In [20]:
6825/(6825+24846)

0.21549682675002368

### **🧮 Default Rate Calculation**

This calculation shows the **default rate** as a percentage:
- **Formula**: Number of defaults ÷ Total loans
- **Interpretation**: What percentage of loans end in default
- **Business Impact**: Higher default rates mean higher risk and potential losses

**Typical Industry Benchmarks:**
- **Prime lending**: 1-3% default rate
- **Subprime lending**: 10-20% default rate  
- **Credit cards**: 2-5% default rate

In [21]:
cr_data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


In [22]:
cr_data.groupby('person_home_ownership').count()['loan_intent']

person_home_ownership
MORTGAGE    13088
OTHER         107
OWN          2410
RENT        16066
Name: loan_intent, dtype: int64

In [23]:
cr_data.groupby('loan_intent').count()['person_home_ownership']

loan_intent
DEBTCONSOLIDATION    5064
EDUCATION            6288
HOMEIMPROVEMENT      3510
MEDICAL              5891
PERSONAL             5365
VENTURE              5553
Name: person_home_ownership, dtype: int64

## 🏠 **Step 6: Categorical Feature Analysis**

### **Understanding Categorical Variables**

Let's explore our categorical features to understand their distributions:

1. **`person_home_ownership`**: Housing situation (RENT, OWN, MORTGAGE, OTHER)
2. **`loan_intent`**: Purpose of the loan (PERSONAL, EDUCATION, MEDICAL, etc.)
3. **`cb_person_default_on_file`**: Previous default history (Y/N)

**Why analyze categorical features?**
- Identify dominant categories
- Check for rare categories that might need special handling
- Understand potential business insights

In [24]:
cr_data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


In [25]:
cr_data_copy=cr_data.drop('loan_grade',axis=1)

### **🗑️ Feature Selection: Removing Loan Grade**

**Why remove `loan_grade`?**

The `loan_grade` feature is often **derived from the target variable** or strongly correlated with default probability. Including it would cause:

- **Data leakage**: Using future information to predict the past
- **Overfitting**: Artificially high model performance
- **Unrealistic results**: Grade might not be available at prediction time

**Best Practice**: Remove features that are:
- Calculated after the target event
- Too closely correlated with the target
- Not available at prediction time

In [26]:
display(cr_data_copy.shape)
display(cr_data_copy.head())

(31671, 11)

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,21,9600,OWN,5.0,EDUCATION,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2


Categorical Features Treatment

## 🔄 **Step 7: Feature Engineering - Categorical Variables**

### **Why Transform Categorical Variables?**

Machine learning algorithms typically work with **numerical data only**. We need to convert categorical text values into numbers:

**Methods Available:**
1. **One-Hot Encoding** (pd.get_dummies): Creates binary columns for each category
2. **Label Encoding**: Assigns numbers to categories (can imply order)
3. **Target Encoding**: Uses target statistics (risk of overfitting)

**Our Approach:**
- **One-hot encoding** for multi-category features (home_ownership, loan_intent)
- **Binary encoding** for yes/no features (default_on_file)

In [27]:
cr_data_cat_treated = cr_data_copy.copy()

In [28]:
person_home_ownership = pd.get_dummies(cr_data_cat_treated['person_home_ownership'],drop_first=True).astype(int)
loan_intent = pd.get_dummies(cr_data_cat_treated['loan_intent'],drop_first=True).astype(int)
cr_data_cat_treated['cb_person_default_on_file_binary'] = np.where(cr_data_cat_treated['cb_person_default_on_file']=='Y',1,0)
cr_data_cat_treated.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,cb_person_default_on_file_binary
0,21,9600,OWN,5.0,EDUCATION,1000,11.14,0,0.10,N,2,0
1,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3,0
2,23,65500,RENT,4.0,MEDICAL,35000,15.23,1,0.53,N,2,0
3,24,54400,RENT,8.0,MEDICAL,35000,14.27,1,0.55,Y,4,1
4,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2,0


### **🎯 One-Hot Encoding Process**

**What's happening here:**

1. **`pd.get_dummies()`**: Converts categorical variables to binary columns
   - **`drop_first=True`**: Removes one column to avoid multicollinearity
   - **`.astype(int)`**: Ensures binary values are integers (0/1)

2. **`np.where()`**: Creates binary variable for default history
   - If 'Y' → 1 (has previous default)
   - If 'N' → 0 (no previous default)

**Result**: Each categorical value becomes a separate binary feature that models can understand.

In [29]:
person_home_ownership.head()

,OTHER,OWN,RENT
0,0,1,0
1,0,0,0
2,0,0,1
3,0,0,1
4,0,1,0


In [30]:
data_to_scale = cr_data_cat_treated.drop(['person_home_ownership','loan_intent','loan_status','cb_person_default_on_file','cb_person_default_on_file_binary'],axis=1)

## ⚖️ **Step 8: Feature Scaling**

### **Why Scale Numerical Features?**

Different features have different ranges:
- **Age**: 18-70 years
- **Income**: $20,000-$150,000  
- **Loan Amount**: $500-$40,000

**Problems without scaling:**
- Models might prioritize features with larger values
- Gradient descent converges slower
- Distance-based algorithms get biased

**StandardScaler Formula:**
```
scaled_value = (original_value - mean) / standard_deviation
```

**Result**: All features have mean=0 and std=1

In [31]:
data_to_scale.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
0,21,9600,5.0,1000,11.14,0.10,2
1,25,9600,1.0,5500,12.87,0.57,3
2,23,65500,4.0,35000,15.23,0.53,2
3,24,54400,8.0,35000,14.27,0.55,4
4,21,9900,2.0,2500,7.14,0.25,2


In [32]:
scaler = StandardScaler()

In [33]:
data_to_scale.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'],
      dtype='object')

In [34]:
scaled_data = scaler.fit_transform(data_to_scale)
scaled_df = pd.DataFrame(scaled_data,columns=['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length'])
scaled_df.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
0,-1.090587,-1.078051,0.054432,-1.367192,0.034115,-0.655113,-0.939656
1,-0.441211,-1.078051,-0.938456,-0.656810,0.597575,3.767461,-0.692664
2,-0.765899,-0.018803,-0.193790,4.000141,1.366226,3.391072,-0.939656
3,-0.603555,-0.229137,0.799097,4.000141,1.053554,3.579267,-0.445671
4,-1.090587,-1.072366,-0.690234,-1.130398,-1.268682,0.756347,-0.939656


### **🔗 Combining Scaled and Categorical Features**

**Final Dataset Preparation:**

1. **Scaled numerical features**: Age, income, employment length, etc.
2. **Binary categorical features**: Home ownership, loan intent dummies  
3. **Target variable**: Loan status (0/1)

**Result**: A complete dataset where:
- All numerical features are standardized (mean=0, std=1)
- All categorical features are binary (0/1)
- Ready for machine learning algorithms

In [35]:

scaled_data_combined = pd.concat([scaled_df,person_home_ownership,loan_intent],axis=1)
scaled_data_combined['cb_person_default_on_file'] = cr_data_cat_treated['cb_person_default_on_file_binary']
scaled_data_combined['loan_status'] = cr_data_cat_treated['loan_status']
scaled_data_combined.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,OTHER,OWN,RENT,EDUCATION,HOMEIMPROVEMENT,MEDICAL,PERSONAL,VENTURE,cb_person_default_on_file,loan_status
0,-1.090587,-1.078051,0.054432,-1.367192,0.034115,-0.655113,-0.939656,0,1,0,1,0,0,0,0,0,0
1,-0.441211,-1.078051,-0.938456,-0.656810,0.597575,3.767461,-0.692664,0,0,0,0,0,1,0,0,0,1
2,-0.765899,-0.018803,-0.193790,4.000141,1.366226,3.391072,-0.939656,0,0,1,0,0,1,0,0,0,1
3,-0.603555,-0.229137,0.799097,4.000141,1.053554,3.579267,-0.445671,0,0,1,0,0,1,0,0,1,1
4,-1.090587,-1.072366,-0.690234,-1.130398,-1.268682,0.756347,-0.939656,0,1,0,0,0,0,0,1,0,1


In [36]:
scaled_data_combined.groupby('loan_status').count()['EDUCATION']

loan_status
0    24846
1     6825
Name: EDUCATION, dtype: int64

In [37]:

target = scaled_data_combined['loan_status']
features = scaled_data_combined.drop('loan_status',axis=1)
features.head()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,OTHER,OWN,RENT,EDUCATION,HOMEIMPROVEMENT,MEDICAL,PERSONAL,VENTURE,cb_person_default_on_file
0,-1.090587,-1.078051,0.054432,-1.367192,0.034115,-0.655113,-0.939656,0,1,0,1,0,0,0,0,0
1,-0.441211,-1.078051,-0.938456,-0.656810,0.597575,3.767461,-0.692664,0,0,0,0,0,1,0,0,0
2,-0.765899,-0.018803,-0.193790,4.000141,1.366226,3.391072,-0.939656,0,0,1,0,0,1,0,0,0
3,-0.603555,-0.229137,0.799097,4.000141,1.053554,3.579267,-0.445671,0,0,1,0,0,1,0,0,1
4,-1.090587,-1.072366,-0.690234,-1.130398,-1.268682,0.756347,-0.939656,0,1,0,0,0,0,0,1,0


Data Balancing

## ⚖️ **Step 9: Handling Class Imbalance with SMOTE**

### **The Class Imbalance Problem**

Our dataset likely has more "good" loans than "defaults":
- **Majority class**: Good loans (0) - ~78%
- **Minority class**: Defaults (1) - ~22%

**Problems with imbalanced data:**
- Model bias toward majority class
- Poor detection of minority class (defaults)
- Misleading accuracy metrics

### **SMOTE Solution**
**Synthetic Minority Oversampling Technique (SMOTE):**
- Creates synthetic examples of minority class
- Uses k-nearest neighbors to generate realistic samples
- Balances the dataset without simple duplication

In [38]:
from imblearn.over_sampling import SMOTE

In [39]:
smote = SMOTE()

In [40]:
balanced_features, balanced_target = smote.fit_resample(features,target)

In [41]:
balanced_target.shape

(49692,)

In [42]:
balanced_target_df = pd.DataFrame({'target':balanced_target})
balanced_target_df.groupby('target').size()

target
0    24846
1    24846
dtype: int64

### **✅ Verifying SMOTE Results**

Let's confirm that SMOTE successfully balanced our dataset:

**Before SMOTE:**
- Unequal class distribution
- Model would be biased toward majority class

**After SMOTE:**
- Equal number of samples for both classes
- Model can learn patterns from both classes equally
- Better detection of default cases

Model training

## 🤖 **Step 10: Machine Learning Model Training**

### **Model Selection Strategy**

We'll train and compare three different algorithms:

1. **Logistic Regression** 📊
   - **Pros**: Interpretable, fast, probabilistic output
   - **Use case**: Baseline model, regulatory compliance
   
2. **Random Forest** 🌳
   - **Pros**: Handles non-linearity, feature importance
   - **Use case**: Complex patterns, ensemble learning
   
3. **XGBoost** ⚡
   - **Pros**: High performance, handles missing values
   - **Use case**: Competitions, complex relationships

### **Train-Test Split**
- **Training set (80%)**: Used to train the model
- **Test set (20%)**: Used to evaluate performance (unseen data)

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [44]:
x_train, x_test, y_train, y_test = train_test_split(balanced_features, balanced_target,test_size=0.20,random_state=42)

In [45]:
logit = LogisticRegression()

In [46]:
logit.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [47]:
logit.score(x_train,y_train)

0.7813246798983724

In [48]:
logit_prediction = logit.predict(x_test)

In [49]:
print(classification_report(y_test,logit_prediction))

              precision    recall  f1-score   support

           0       0.78      0.78      0.78      4995
           1       0.78      0.78      0.78      4944

    accuracy                           0.78      9939
   macro avg       0.78      0.78      0.78      9939
weighted avg       0.78      0.78      0.78      9939



### **📈 Model 1: Logistic Regression**

**Why start with Logistic Regression?**

- **Industry standard** for credit risk modeling
- **Regulatory approved** by banking authorities
- **Interpretable coefficients** show feature impact
- **Probabilistic output** directly gives default probability
- **Fast training** and prediction
- **Baseline model** to compare others against

**How it works:**
- Uses linear combination of features
- Applies sigmoid function to get probabilities
- Coefficients show positive/negative impact on default risk

In [50]:
print(logit.coef_[0])

[-0.07079387  0.07555523 -0.05984562 -0.6670677   0.9895237   1.405714
  0.01404678 -0.64465753 -2.02668148  0.46881988 -1.26075203 -0.34243505
 -0.54855143 -1.02599919 -1.52510956  0.08438428]


In [51]:
features_imp_logit = pd.DataFrame({'features':balanced_features.columns,'logit_imp':logit.coef_[0]})

In [52]:
features_imp_logit.sort_values(by='logit_imp',ascending=False)

,features,logit_imp
5,loan_percent_income,1.405714
4,loan_int_rate,0.989524
9,RENT,0.468820
15,cb_person_default_on_file,0.084384
1,person_income,0.075555
6,cb_person_cred_hist_length,0.014047
2,person_emp_length,-0.059846
0,person_age,-0.070794
11,HOMEIMPROVEMENT,-0.342435
12,MEDICAL,-0.548551


### **🔍 Logistic Regression Feature Importance**

**Understanding Coefficients:**

- **Positive coefficients**: Increase probability of default
- **Negative coefficients**: Decrease probability of default  
- **Larger absolute values**: Stronger impact on prediction

**Business Interpretation:**
- Shows which factors most influence default risk
- Helps identify key risk drivers
- Supports regulatory explanations
- Guides business policy decisions

## Randomforest Model


### **🌳 Model 2: Random Forest**

**Why Random Forest?**

- **Ensemble method**: Combines multiple decision trees
- **Handles non-linearity**: Captures complex patterns
- **Feature importance**: Shows which variables matter most
- **Robust to outliers**: Less sensitive to extreme values
- **Reduces overfitting**: Averaging reduces variance

**How it works:**
- Trains multiple decision trees on random subsets
- Each tree votes on the final prediction
- Averages predictions for final result
- Provides built-in feature importance scores

In [53]:
rf=RandomForestClassifier()


In [54]:
rf.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [55]:
rf.score(x_train,y_train)

1.0

In [56]:
rf_prediction = rf.predict(x_test)
rf_prediction

array([0, 0, 0, ..., 0, 0, 1], shape=(9939,))

In [57]:
print(classification_report(y_test,rf_prediction))

              precision    recall  f1-score   support

           0       0.91      0.97      0.94      4995
           1       0.97      0.91      0.94      4944

    accuracy                           0.94      9939
   macro avg       0.94      0.94      0.94      9939
weighted avg       0.94      0.94      0.94      9939



In [58]:
rf.feature_importances_

array([0.0628057 , 0.14992226, 0.07361983, 0.07923583, 0.21003386,
       0.20839938, 0.05946032, 0.00045909, 0.02027557, 0.0480822 ,
       0.01496518, 0.01690003, 0.00874864, 0.01316371, 0.01714763,
       0.01678077])

In [59]:
features_imp_rf = pd.DataFrame({'features':balanced_features.columns,'rf_imp':rf.feature_importances_})
features_imp_rf.sort_values(by='rf_imp',ascending=False)

,features,rf_imp
4,loan_int_rate,0.210034
5,loan_percent_income,0.208399
1,person_income,0.149922
3,loan_amnt,0.079236
2,person_emp_length,0.073620
0,person_age,0.062806
6,cb_person_cred_hist_length,0.059460
9,RENT,0.048082
8,OWN,0.020276
14,VENTURE,0.017148


## XGboost Model

### **⚡ Model 3: XGBoost**

**Why XGBoost (Extreme Gradient Boosting)?**

- **State-of-the-art performance**: Often wins competitions
- **Gradient boosting**: Learns from previous model mistakes
- **Handles missing values**: Built-in missing value treatment
- **Regularization**: Prevents overfitting
- **Feature importance**: Multiple importance metrics

**How it works:**
- Builds models sequentially
- Each new model corrects errors of previous models  
- Combines weak learners into strong predictor
- Uses gradient descent optimization

In [60]:
xgb_model = XGBClassifier(tree_method='exact')



In [61]:
xgb_model.fit(x_train,y_train.values.ravel())


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [62]:

xgb_model.score(x_train,y_train.values.ravel())

0.9651095514803915

In [63]:
xgb_prediction = xgb_model.predict(x_test)
xgb_prediction

array([0, 0, 0, ..., 0, 0, 1], shape=(9939,))

In [64]:
print(classification_report(y_test,xgb_prediction))

              precision    recall  f1-score   support

           0       0.92      0.98      0.95      4995
           1       0.98      0.91      0.95      4944

    accuracy                           0.95      9939
   macro avg       0.95      0.95      0.95      9939
weighted avg       0.95      0.95      0.95      9939



In [65]:
features_imp_rf = pd.DataFrame({'features':balanced_features.columns,'rf_imp':rf.feature_importances_})
features_imp_xgb = pd.DataFrame({'features':balanced_features.columns,'xgb_imp':xgb_model.feature_importances_})
features_imp_xgb.sort_values(by='xgb_imp',ascending=False)

,features,xgb_imp
8,OWN,0.159804
5,loan_percent_income,0.138316
9,RENT,0.118236
14,VENTURE,0.087755
6,cb_person_cred_hist_length,0.073452
4,loan_int_rate,0.071373
10,EDUCATION,0.065927
11,HOMEIMPROVEMENT,0.054935
2,person_emp_length,0.050453
13,PERSONAL,0.046090


In [66]:
features_imp = pd.concat([features_imp_logit,features_imp_rf,features_imp_xgb],axis=1)
features_imp

,features,logit_imp,features,rf_imp,features,xgb_imp
0,person_age,-0.070794,person_age,0.062806,person_age,0.039959
1,person_income,0.075555,person_income,0.149922,person_income,0.037744
2,person_emp_length,-0.059846,person_emp_length,0.073620,person_emp_length,0.050453
3,loan_amnt,-0.667068,loan_amnt,0.079236,loan_amnt,0.010298
4,loan_int_rate,0.989524,loan_int_rate,0.210034,loan_int_rate,0.071373
5,loan_percent_income,1.405714,loan_percent_income,0.208399,loan_percent_income,0.138316
6,cb_person_cred_hist_length,0.014047,cb_person_cred_hist_length,0.059460,cb_person_cred_hist_length,0.073452
7,OTHER,-0.644658,OTHER,0.000459,OTHER,0.009454
8,OWN,-2.026681,OWN,0.020276,OWN,0.159804
9,RENT,0.468820,RENT,0.048082,RENT,0.118236


## 📊 **Step 11: Model Comparison & Feature Importance**

### **Comparing All Models**

This comprehensive comparison shows:
- **Feature rankings** across different algorithms
- **Consensus features**: Important across all models  
- **Algorithm-specific insights**: What each model prioritizes
- **Ensemble potential**: Could combine model strengths

**Key Questions:**
- Which features are consistently important?
- Do models agree on risk factors?
- Which algorithm captures the best patterns?

Nest step Exploration

In [67]:
xgb_prediction_df = pd.DataFrame({'test_indices_xgb':x_test.index,'xgb_pred':xgb_prediction})
rf_prediction_df = pd.DataFrame({'test_indices_rf':x_test.index,'rf_pred':rf_prediction})
logit_prediction_df = pd.DataFrame({'test_indices_logit':x_test.index,'logit_pred':logit_prediction})

In [68]:
xgb_prediction_df.head()

,test_indices_xgb,xgb_pred
0,24808,0
1,9935,0
2,14054,0
3,147,0
4,4070,1


In [69]:
merged_with_orig = cr_data_copy.merge(xgb_prediction_df,left_index=True,right_on='test_indices_xgb',how='left')
merged_with_orig.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,test_indices_xgb,xgb_pred
NaN,21,9600,OWN,5.0,EDUCATION,1000,11.14,0,0.10,N,2,0,NaN
9831.0,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3,1,1.0
NaN,23,65500,RENT,4.0,MEDICAL,35000,15.23,1,0.53,N,2,2,NaN
NaN,24,54400,RENT,8.0,MEDICAL,35000,14.27,1,0.55,Y,4,3,NaN
442.0,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2,4,1.0


In [70]:
merged_with_rf = merged_with_orig.merge(rf_prediction_df,left_index=True,right_on='test_indices_rf',how='left')
merged_with_rf.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,test_indices_xgb,xgb_pred,test_indices_rf,rf_pred
NaN,21,9600,OWN,5.0,EDUCATION,1000,11.14,0,0.10,N,2,0,NaN,NaN,NaN
NaN,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3,1,1.0,9831.0,NaN
NaN,23,65500,RENT,4.0,MEDICAL,35000,15.23,1,0.53,N,2,2,NaN,NaN,NaN
NaN,24,54400,RENT,8.0,MEDICAL,35000,14.27,1,0.55,Y,4,3,NaN,NaN,NaN
NaN,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2,4,1.0,442.0,NaN


In [71]:
merged_with_final = merged_with_rf.merge(logit_prediction_df,left_index=True,right_on='test_indices_logit',how='left')
merged_with_final.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,test_indices_xgb,xgb_pred,test_indices_rf,rf_pred,test_indices_logit,logit_pred
NaN,21,9600,OWN,5.0,EDUCATION,1000,11.14,0,0.10,N,2,0,NaN,NaN,NaN,NaN,NaN
NaN,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,1,0.57,N,3,1,1.0,9831.0,NaN,NaN,NaN
NaN,23,65500,RENT,4.0,MEDICAL,35000,15.23,1,0.53,N,2,2,NaN,NaN,NaN,NaN,NaN
NaN,24,54400,RENT,8.0,MEDICAL,35000,14.27,1,0.55,Y,4,3,NaN,NaN,NaN,NaN,NaN
NaN,21,9900,OWN,2.0,VENTURE,2500,7.14,1,0.25,N,2,4,1.0,442.0,NaN,NaN,NaN


In [72]:
merged_with_final.shape

(31671, 17)

In [73]:
merged_with_final.dropna(inplace=True)

In [79]:
merged_with_final.to_csv("D:\Creditrisk_ENDtoEND\prepareddata\prepareddata.csv",index=False)

In [80]:
final_data_with_pred = merged_with_final.drop(['test_indices_xgb','test_indices_rf','test_indices_logit'],axis=1)
final_data_with_pred.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,xgb_pred,rf_pred,logit_pred
7782.0,24,78956,RENT,5.0,MEDICAL,35000,11.11,1,0.44,N,4,1.0,0.0,1.0
7968.0,25,12600,OWN,3.0,PERSONAL,1750,13.61,0,0.14,N,3,0.0,1.0,0.0
9251.0,22,153000,MORTGAGE,5.0,DEBTCONSOLIDATION,24000,15.62,1,0.16,Y,2,1.0,1.0,0.0
5662.0,22,16094,MORTGAGE,2.0,VENTURE,5500,7.14,1,0.34,N,3,1.0,0.0,1.0
143.0,21,18000,OWN,1.0,PERSONAL,2500,11.36,0,0.14,N,3,0.0,1.0,0.0


In [81]:
final_data_with_pred.to_excel(r"D:\Creditrisk_ENDtoEND\prepareddata\pd_prediction.xlsx",index=False)